# GNR 607 - Morphological opening with growing structuring elements
Run the cells **top to bottom** (Shift+Enter on each, or Runtime > Run all).

## Step 1 - Get the project from GitHub
Clones the repository into `/content/GNR607` (no upload needed).

In [ ]:
!git clone -q https://github.com/PiyushTailor1905/gnr607-morphological-opening /content/GNR607
!ls /content/GNR607 /content/GNR607/02_Source_Code

## Step 2 - The two input satellite images (Sentinel-2, 10 m/pixel, 1024x1024)

In [ ]:
import cv2, matplotlib.pyplot as plt
D = "/content/GNR607/04_Sample_Data"
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
for a, (f, t) in zip(ax, [("urban_image/urban_image.png", "Urban (central Delhi)"),
                          ("smooth_landscape_image/smooth_landscape_image.png", "Smooth (reservoir + forest)")]):
    a.imshow(cv2.imread(f"{D}/{f}")[..., ::-1]); a.set_title(t); a.axis("off")
plt.show()

## Step 3 - Binary image: grayscale + Otsu threshold (live)

In [ ]:
import sys
sys.path.insert(0, "/content/GNR607/02_Source_Code")
import numpy as np
from image_loader import load_image
from preprocessing import to_gray, binarize

paths = {"Urban": f"{D}/urban_image/urban_image.png", "Smooth": f"{D}/smooth_landscape_image/smooth_landscape_image.png"}
binary = {}
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for r, (name, p) in enumerate(paths.items()):
    gray = to_gray(load_image(p))
    A, T = binarize(gray)                      # Otsu; foreground = gray > T
    binary[name] = A
    print(f"{name}: Otsu threshold T = {T}, foreground = {A.sum():,} px ({100*A.mean():.2f} %)")
    ax[r, 0].imshow(gray, cmap="gray"); ax[r, 0].set_title(f"{name}: gray"); ax[r, 0].axis("off")
    ax[r, 1].hist(gray.ravel(), 256, color="gray"); ax[r, 1].axvline(T, color="r"); ax[r, 1].set_title(f"histogram, T={T}")
    ax[r, 2].imshow(A, cmap="gray"); ax[r, 2].set_title(f"{name}: binary"); ax[r, 2].axis("off")
plt.tight_layout(); plt.show()

## Step 4 - Structuring elements grow by dilation with the 3x3 box

In [ ]:
from morphology import box_se, se_sequence, dilate_se
fig, ax = plt.subplots(1, 5, figsize=(15, 3.6))
for (k, se), a in zip(se_sequence(box_se(3)), ax):
    n = se.shape[0]
    a.imshow(se, cmap="Blues", vmin=0, vmax=1.6, interpolation="nearest")   # blue = element pixel present
    a.set_xticks(np.arange(-.5, n, 1), minor=True); a.set_yticks(np.arange(-.5, n, 1), minor=True)
    a.grid(which="minor", color="black", lw=1)
    a.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
    a.set_title(f"SE{k}: {n}x{n}")
plt.show()
se = box_se(3)
for _ in range(3):
    se = dilate_se(se, box_se(3)); print("dilate by 3x3 ->", se.shape, "all ones:", bool(se.all()))

## Step 5 - THE EXPERIMENT, live: open the ORIGINAL image with each SE and count

In [ ]:
from morphology import opening, count_foreground
results = {}
for name, A in binary.items():
    sizes, counts = [], []
    for k, se in se_sequence(box_se(3)):
        n = count_foreground(opening(A, se))   # opening = erosion then dilation, same SE, applied to the original A
        sizes.append(se.shape[0]); counts.append(n)
        print(f"{name:6s} SE {se.shape[0]:3d}x{se.shape[0]:<3d} foreground = {n:7d}")
        if n == 0: break                        # null image -> stop
    results[name] = (sizes, counts)
    print()

## Step 6 - The required graphs

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, (name, col) in zip(ax[:2], [("Urban", "C3"), ("Smooth", "C0")]):
    s, c = results[name]
    a.plot(s, c, "-o", color=col, ms=4); a.set_title(f"{name}: foreground pixels remaining vs SE size")
    a.set_xlabel("Structuring element size (side, pixels)"); a.set_ylabel("Foreground pixels after opening"); a.grid(alpha=.3)
for name, col in [("Urban", "C3"), ("Smooth", "C0")]:
    s, c = results[name]; ax[2].plot(s, 100 * np.array(c) / binary[name].sum(), "-o", color=col, ms=3, label=name)
ax[2].set_title("Both, as % of original foreground"); ax[2].set_xlabel("SE size"); ax[2].set_ylabel("% remaining"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Step 7 - Opened images at selected sizes

In [ ]:
ses = {se.shape[0]: se for (k, se), _ in zip(se_sequence(box_se(3)), range(30))}
fig, ax = plt.subplots(2, 5, figsize=(20, 8.5))
for r, name in enumerate(binary):
    for a, s in zip(ax[r], [None, 3, 9, 17, 21 if name == "Urban" else 49]):
        img = binary[name] if s is None else opening(binary[name], ses[s])
        a.imshow(img, cmap="gray"); a.axis("off"); a.set_title(f"{name}: " + ("binary input" if s is None else f"SE {s}x{s}, N={img.sum():,}"))
plt.tight_layout(); plt.show()

## Step 8 - Run the project's own full pipeline (all images, tables, comparison)

In [ ]:
%cd /content/GNR607/02_Source_Code
!python main.py

In [ ]:
import pandas as pd
from IPython.display import Image, display
R = "/content/GNR607/05_Results"
for name in ["Urban", "Smooth_Landscape"]:
    df = pd.read_csv(f"{R}/{name}/results.csv"); print(name); display(df.drop(columns="opened_file"))
    display(Image(f"{R}/{name}/curve.png", width=900))
display(Image(f"{R}/Comparative_Analysis/comparison_graph.png", width=1200))
print(open(f"{R}/Comparative_Analysis/comparison_summary.txt").read())

## Step 9 - Independent validation (about 1-2 minutes)

In [ ]:
!python validate.py | tail -25

## Step 1 - Get the project from GitHub
Clones the repository into `/content/GNR607` (no upload needed).

In [ ]:
!PYTHONPATH=. python sensitivity.py | head -20